# Examen - Python avancé
Emma Loquet, M1 ECAP

<br>

### Importation des bibliothèques

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [2]:
#On définit un style par défault
pio.templates.default = "plotly_white"

### Préparation des données

In [3]:
# Chargement de `supermarket_sales.csv` dans un DataFrame `df`.
df = pd.read_csv("datasets/supermarket_sales.csv")

df.head()

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,13:08,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,13:23,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,20:33,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37,Ewallet,604.17,4.761905,30.2085,5.3


In [4]:
# On conserve uniquement les colonnes utiles
df = df[['Invoice ID', 'Gender', 'City', 'Product line', 'Quantity', 'Unit price', 'Tax 5%', 'Total', 'Date', 'Rating']]
df.head()

,Invoice ID,Gender,City,Product line,Quantity,Unit price,Tax 5%,Total,Date,Rating
0,750-67-8428,Female,Yangon,Health and beauty,7,74.69,26.1415,548.9715,1/5/2019,9.1
1,226-31-3081,Female,Naypyitaw,Electronic accessories,5,15.28,3.8200,80.2200,3/8/2019,9.6
2,631-41-3108,Male,Yangon,Home and lifestyle,7,46.33,16.2155,340.5255,3/3/2019,7.4
3,123-19-1176,Male,Yangon,Health and beauty,8,58.22,23.2880,489.0480,1/27/2019,8.4
4,373-73-7910,Male,Yangon,Sports and travel,7,86.31,30.2085,634.3785,2/8/2019,5.3


In [5]:
df.info()
# Il n'y a aucune valeur manquante dans note base de données

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Invoice ID    1000 non-null   object 
 1   Gender        1000 non-null   object 
 2   City          1000 non-null   object 
 3   Product line  1000 non-null   object 
 4   Quantity      1000 non-null   int64  
 5   Unit price    1000 non-null   float64
 6   Tax 5%        1000 non-null   float64
 7   Total         1000 non-null   float64
 8   Date          1000 non-null   object 
 9   Rating        1000 non-null   float64
dtypes: float64(4), int64(1), object(5)
memory usage: 78.2+ KB


In [6]:
#On change le type de la colonne 'Date' pour que ce soit bien une date
df['Date'] = pd.to_datetime(df['Date'])

df.info()  #Les changements ont bien été effectués

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Invoice ID    1000 non-null   object        
 1   Gender        1000 non-null   object        
 2   City          1000 non-null   object        
 3   Product line  1000 non-null   object        
 4   Quantity      1000 non-null   int64         
 5   Unit price    1000 non-null   float64       
 6   Tax 5%        1000 non-null   float64       
 7   Total         1000 non-null   float64       
 8   Date          1000 non-null   datetime64[ns]
 9   Rating        1000 non-null   float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(4)
memory usage: 78.2+ KB


### Création des fonctions

In [7]:
def montant_total_achats(data) :
    MT = data['Total'].sum()
    return MT

print("Le montant total d'achat est :", montant_total_achats(df), "€")

Le montant total d'achat est : 322966.749 €


In [8]:
def evaluation_moyenne(data) :
    moyenne = data['Rating'].mean()
    return moyenne

print("La moyenne des évaluations est :", evaluation_moyenne(df))

La moyenne des évaluations est : 6.9727


In [9]:
def nombre_total_achats(data) :
    total = data['Invoice ID'].count()
    return total

print("Le nombre total d'achat est :", nombre_total_achats(df))

Le nombre total d'achat est : 1000


### Création des graphiques

In [10]:
def plot_evolution_montant_total(data) :

    #Montant total des achats par semaine et par ville
    evolution = (
        data
        .groupby([pd.Grouper(key="Date", freq="W"), "City"])["Total"]
        .sum()
        .reset_index()
    )


    fig = px.line(
        evolution,
        x='Date',
        y='Total',
        color="City",
        color_discrete_sequence=["#2E86AB", "#F18F01", "#6C757D"],
        title="Évolution du montant total des achats par semaine et par ville",
        labels={
            "Date" : "Semaine",
            "Total" : "Montant total des achats",
            "City" : "Ville"
        }
    )

    fig.update_layout(
    template="plotly_white",  # style plus propre
    title_x=0.5           # centre le titre
    )

    return fig

plot_evolution_montant_total(df)

In [11]:
def barplot_nb_total_achats(data):

    df = (
        data
        .groupby(['City', 'Gender'])['Invoice ID']
        .count()
        .reset_index(name='Nb_Achats')
    )

    fig = px.bar(
        df,
        x='Nb_Achats',
        y='City',
        color='Gender',
        color_discrete_sequence=["#2E86AB", "#1B4F72"],
        barmode='group',
        title="Nombre total d'achats par ville et par sexe",
        labels={
            "Nb_Achats": "Nombre d'achats",
            "City": "Ville",
            "Gender": "Sexe"
        }
    )

    fig.update_layout(
        template="plotly_white",
        margin=dict(l=20, r=20, t=40, b=60),
        title_x=0.5,
        bargap=0.45
    )

    
    return fig

barplot_nb_total_achats(df)

In [12]:
def pie_product_line(data):

    df = (
        data
        .groupby("Product line")
        .size()
        .reset_index(name="Part_Achat")
    )

    fig = px.pie(
        df,
        names="Product line",
        values="Part_Achat",
        title="Répartition des catégories de produits",
        color_discrete_sequence=[
            "#1B4F72",  # bleu foncé
            "#2E86AB",  # bleu principal
            "#5DADE2",  # bleu moyen
            "#85C1E9",  # bleu clair
            "#AED6F1",  # très clair
            "#D6EAF8"]
    )

    fig.update_layout(template="plotly_white", 
                      title_x=0.5,
                      margin=dict(l=10, r=10, t=40, b=10))
    

    return fig

pie_product_line(df)

### Tableau de bord

In [13]:
import dash
from dash import dcc, html, dash_table, Input, Output
import dash_bootstrap_components as dbc

app = dash.Dash(external_stylesheets=[dbc.themes.BOOTSTRAP])
server = app.server

liste_villes = [
    {"label": v, "value": v}
    for v in df["City"].dropna().unique()
]

liste_sexe = [
    {"label": g, "value": g}
    for g in df["Gender"].dropna().unique()
]


app.layout = dbc.Container([

    # HEADER
    dbc.Row([
        dbc.Col(html.H3("Analyse des ventes - Supermarché", style={"fontSize": "22px","fontWeight": "bold"}), md=4),
        dbc.Col(
            dcc.Dropdown(
                id="city-dropdown",
                options=liste_villes,
                placeholder="Ville",
                style={"color": "black"}
            ),
            md=4
        ),

        dbc.Col(
            dcc.Dropdown(
                id="gender-dropdown",
                options=liste_sexe,
                placeholder="Sexe",
                style={"color": "black"}
            ),
            md=4
        )
    ], style={"backgroundColor": "#2E86AB", "padding": "10px", "color": "white"}),

    html.Br(),

    # KPI
    dbc.Row([

        dbc.Col(html.Div(id="kpi_total", style={"fontSize": "20px", "fontWeight": "bold", "textAlign": "center"}), md=4),
        dbc.Col(html.Div(id="kpi_nb", style={"fontSize": "20px", "fontWeight": "bold", "textAlign": "center"}), md=4),
        dbc.Col(html.Div(id="kpi_rating", style={"fontSize": "20px", "fontWeight": "bold", "textAlign": "center"}), md=4),

    ], style={"marginBottom": "20px"}),

    html.Hr(),

    # GRAPHIQUES
    dbc.Row([

        dbc.Col(dcc.Graph(id="fig_line", style={"height": "350px"}), md=7),
        dbc.Col(dcc.Graph(id="fig_bar", style={"height": "350px"}), md=5),

    ], style={"marginBottom": "20px"}),

    dbc.Row([
        dbc.Col(dcc.Graph(id="fig_pie", style={"height": "300px"}), md=6),
        dbc.Col([
        html.H5("Tableau des derniers achats", style={"marginBottom": "10px","fontWeight": "normal", "fontSize": "20px"}),
        dash_table.DataTable(
            id="table_ventes",
            page_size=8,
            style_table={"overflowX": "auto"},
            style_cell={"fontSize": "12px", "textAlign": "left"}
        )
    ], md=6)
    ], style={"marginBottom": "20px"})

], fluid=True)


@app.callback(
    Output("kpi_total", "children"),
    Output("kpi_nb", "children"),
    Output("kpi_rating", "children"),
    Output("fig_line", "figure"),
    Output("fig_bar", "figure"),
    Output("fig_pie", "figure"),
    Output("table_ventes", "data"),      
    Output("table_ventes", "columns"),
    Input("city-dropdown", "value"),
    Input("gender-dropdown", "value")
)

def update_dashboard(city, gender):

    df_filter = df.copy()

    if city:
        df_filter = df_filter[df_filter["City"] == city]

    if gender:
        df_filter = df_filter[df_filter["Gender"] == gender]

    # KPI
    total = montant_total_achats(df_filter)
    nb = nombre_total_achats(df_filter)
    rating = evaluation_moyenne(df_filter)

    kpi_total = f"Montant total : {round(total, 2)} €"
    kpi_nb = f"Nombre d'achats : {nb}"
    kpi_rating = f"Note moyenne : {round(rating, 2)}"

    fig_line = plot_evolution_montant_total(df_filter)
    fig_bar = barplot_nb_total_achats(df_filter)
    fig_pie = pie_product_line(df_filter)

    df_table = df_filter.sort_values("Date", ascending=False).head(100)
    data_table = df_table.to_dict("records")
    columns = [{"name": c, "id": c} for c in df_table.columns]

    return kpi_total, kpi_nb, kpi_rating, fig_line, fig_bar, fig_pie, data_table, columns


if __name__ == '__main__' :
    app.run(debug=True, port=8031, jupyter_mode="external")

Dash app running on http://127.0.0.1:8031/
